# Stage 2: Building my PMC corpus

This notebook is the next stage of my Agentic AI-RAG dissertation project. My first prototype used six synthetic records to check the workflow. I am now moving towards a real research corpus about AI in healthcare.

I use the official NCBI E-Utilities service rather than downloading articles from the main PMC website. I also restrict the search to the PMC Open Access Subset with a CC BY licence. The output is a corpus manifest, a screening log and a JSONL file containing the selected article text.

**Important:** the automatic selection is only a candidate corpus. I still need to read the titles and abstracts and record my manual inclusion or exclusion decision before calling it the final dissertation corpus.

## 1. My selection plan

I am aiming for a balanced corpus of approximately 50 articles: ten articles for each of five themes.

### Inclusion rules

- Published between 2020 and 2026
- Written in English
- Available in the PMC Open Access Subset
- Machine-readable CC BY licence
- Relevant to AI in healthcare
- At least 800 words of extractable full text

### Exclusion rules

- Retraction, correction or expression of concern
- Duplicate article
- Not clearly related to the assigned theme
- Missing or unclear reusable licence
- Insufficient extractable text

I use relevance ranking to create the candidate list, but I will not treat relevance ranking as a replacement for manual screening.

## 2. Libraries and settings

The code below uses standard Python libraries. NCBI asks software users to identify their tool and provide an email address. The email is used only as an E-Utilities request parameter.

In [ ]:
# Standard Python libraries used for API requests, XML parsing and files.
import csv
import json
import re
import shutil
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path

# I enter my university email when the notebook runs.
NCBI_EMAIL = input("Enter your university email for the NCBI request: " ).strip()
if "@" not in NCBI_EMAIL:
    raise ValueError("Please enter a valid email address before continuing.")

TOOL_NAME = "agentic_ai_dissertation"
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
TARGET_PER_THEME = 10
CANDIDATES_PER_THEME = 15
SEARCH_RESULTS_PER_THEME = 40
MINIMUM_WORDS = 800

# NCBI permits up to three requests per second without an API key.
REQUEST_DELAY_SECONDS = 0.4
RETRIEVAL_DATE = datetime.now(timezone.utc).date().isoformat()
OUTPUT_DIRECTORY = Path("/content/pmc_corpus_stage2")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Settings ready. Retrieval date:", RETRIEVAL_DATE)

## 3. Five search themes

I separated the broad topic into five themes so that one popular application does not dominate the corpus. Every query also applies the date, English-language and CC BY filters.

In [ ]:
AI_TERMS = '("artificial intelligence"[Title/Abstract] OR "machine learning"[Title/Abstract] OR "deep learning"[Title/Abstract] OR "large language model"[Title/Abstract])'
HEALTH_TERMS = '(healthcare[Title/Abstract] OR medical[Title/Abstract] OR clinical[Title/Abstract] OR medicine[Title/Abstract])'
COMMON_FILTERS = 'AND "cc by license"[filter] AND english[language] AND 2020:2026[pubdate]'

THEME_TERMS = {
    "medical_imaging_and_diagnosis": '("medical imaging"[Title/Abstract] OR diagnosis[Title/Abstract] OR diagnostic[Title/Abstract])',
    "clinical_decision_support": '("clinical decision support"[Title/Abstract] OR treatment[Title/Abstract] OR prognosis[Title/Abstract])',
    "healthcare_operations": '(workflow[Title/Abstract] OR administration[Title/Abstract] OR "resource allocation"[Title/Abstract] OR operational[Title/Abstract])',
    "generative_ai_and_llms": '("large language model"[Title/Abstract] OR "generative AI"[Title/Abstract] OR ChatGPT[Title/Abstract])',
    "ethics_safety_and_bias": '(bias[Title/Abstract] OR fairness[Title/Abstract] OR privacy[Title/Abstract] OR ethics[Title/Abstract] OR hallucination[Title/Abstract] OR safety[Title/Abstract])',
}

SEARCH_QUERIES = {
    theme: f"{AI_TERMS} AND {HEALTH_TERMS} AND {theme_terms} {COMMON_FILTERS}"
    for theme, theme_terms in THEME_TERMS.items()
}

for theme, query in SEARCH_QUERIES.items():
    print("\nTHEME:", theme)
    print(query)

## 4. Official NCBI request functions

The helper below adds my tool name and email to every request, waits between calls and retries temporary errors. The search function returns PMC identifiers, and the fetch function retrieves article XML in small batches.

In [ ]:
def ncbi_request(endpoint, parameters, attempts=3):
    """Send a polite, identified request to an official NCBI endpoint."""
    request_parameters = dict(parameters)
    request_parameters["tool"] = TOOL_NAME
    request_parameters["email"] = NCBI_EMAIL
    url = f"{EUTILS_BASE}/{endpoint}?{urllib.parse.urlencode(request_parameters)}"
    request = urllib.request.Request(
        url,
        headers={"User-Agent": f"{TOOL_NAME}/0.2 ({NCBI_EMAIL})"},
    )

    for attempt in range(1, attempts + 1):
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                data = response.read()
            time.sleep(REQUEST_DELAY_SECONDS)
            return data
        except (urllib.error.URLError, TimeoutError) as error:
            if attempt == attempts:
                raise RuntimeError(f"NCBI request failed after {attempts} attempts") from error
            time.sleep(2 ** attempt)


def search_pmc(query, maximum_results=40):
    """Search PMC and return relevance-ranked numeric PMC identifiers."""
    data = ncbi_request(
        "esearch.fcgi",
        {
            "db": "pmc",
            "term": query,
            "retmode": "json",
            "retmax": maximum_results,
            "sort": "relevance",
        },
    )
    result = json.loads(data.decode("utf-8"))["esearchresult"]
    return result["idlist"], int(result["count"])


def fetch_pmc_xml(pmc_ids):
    """Retrieve full article XML for a small list of PMC identifiers."""
    return ncbi_request(
        "efetch.fcgi",
        {
            "db": "pmc",
            "id": ",".join(pmc_ids),
            "retmode": "xml",
        },
    )

## 5. Search and create a balanced candidate list

I retain up to 15 unique candidates per theme. If the same article appears in more than one search, I keep its first assigned theme and record the duplicate rather than downloading it twice.

In [ ]:
search_log = []
candidate_ids_by_theme = {}
already_assigned = set()

for theme, query in SEARCH_QUERIES.items():
    result_ids, total_results = search_pmc(query, SEARCH_RESULTS_PER_THEME)
    unique_ids = []
    duplicate_count = 0

    for pmc_id in result_ids:
        if pmc_id in already_assigned:
            duplicate_count += 1
            continue
        unique_ids.append(pmc_id)
        already_assigned.add(pmc_id)
        if len(unique_ids) == CANDIDATES_PER_THEME:
            break

    candidate_ids_by_theme[theme] = unique_ids
    search_log.append({
        "theme": theme,
        "query": query,
        "total_matching_records": total_results,
        "returned_records": len(result_ids),
        "retained_unique_candidates": len(unique_ids),
        "duplicates_skipped": duplicate_count,
        "retrieval_date": RETRIEVAL_DATE,
    })
    print(theme, "->", len(unique_ids), "unique candidates")

all_candidate_ids = [
    pmc_id
    for theme_ids in candidate_ids_by_theme.values()
    for pmc_id in theme_ids
]
print("Total unique candidates:", len(all_candidate_ids))

## 6. Parse article metadata, licence and text

PMC returns structured XML. I extract the identifiers, title, authors, journal, year, licence, abstract and body text. I also record the article type so that retractions and corrections can be excluded.

In [ ]:
XLINK_HREF = "{http://www.w3.org/1999/xlink}href"


def clean_text(value):
    """Remove repeated spaces while preserving readable text."""
    return re.sub(r"\s+", " ", value or "").strip()


def element_text(element):
    """Collect text even when the XML element contains nested tags."""
    return clean_text(" ".join(element.itertext())) if element is not None else ""


def find_article_id(article, id_type):
    element = article.find(f".//article-id[@pub-id-type='{id_type}']")
    return element_text(element)


def identify_cc_licence(licence_url, licence_text):
    """Convert the licence URL/text into a consistent CC label."""
    value = f"{licence_url} {licence_text}".lower()
    if "creativecommons.org/publicdomain/zero" in value or "cc0" in value:
        return "CC0"
    if "by-nc-nd" in value or ("noncommercial" in value and "noderivatives" in value):
        return "CC BY-NC-ND"
    if "by-nc-sa" in value or ("noncommercial" in value and "sharealike" in value):
        return "CC BY-NC-SA"
    if "by-nc" in value or "noncommercial" in value:
        return "CC BY-NC"
    if "by-nd" in value or "noderivatives" in value or "no derivatives" in value:
        return "CC BY-ND"
    if "by-sa" in value or "sharealike" in value or "share alike" in value:
        return "CC BY-SA"
    if "creativecommons.org/licenses/by/" in value or "creative commons attribution" in value:
        return "CC BY"
    return "UNCONFIRMED"


def parse_article(article):
    """Convert one JATS article element into a research-corpus record."""
    # Current PMC XML normally labels this identifier as 'pmcid'.
    # The fallbacks keep the parser compatible with older records.
    pmcid = (
        find_article_id(article, "pmcid")
        or find_article_id(article, "pmc")
        or find_article_id(article, "pmcaid")
    )
    if pmcid and not pmcid.upper().startswith("PMC"):
        pmcid = f"PMC{pmcid}"

    title = element_text(article.find(".//article-title"))
    journal = element_text(article.find(".//journal-title"))

    year_element = article.find(".//pub-date[@pub-type='epub']/year")
    if year_element is None:
        year_element = article.find(".//pub-date/year")
    year = element_text(year_element)

    author_names = []
    for name in article.findall(".//contrib[@contrib-type='author']/name"):
        given = element_text(name.find("given-names"))
        surname = element_text(name.find("surname"))
        full_name = clean_text(f"{given} {surname}")
        if full_name:
            author_names.append(full_name)

    licence_element = article.find(".//license")
    licence_url = licence_element.get(XLINK_HREF, "") if licence_element is not None else ""
    licence_text = element_text(licence_element)
    licence = identify_cc_licence(licence_url, licence_text)

    abstract = element_text(article.find(".//abstract"))
    body_paragraphs = [
        element_text(paragraph)
        for paragraph in article.findall(".//body//p")
    ]
    body_paragraphs = [text for text in body_paragraphs if text]
    full_text = "\n\n".join(body_paragraphs)

    return {
        "pmcid": pmcid,
        "pmid": find_article_id(article, "pmid"),
        "doi": find_article_id(article, "doi"),
        "title": title,
        "authors": "; ".join(author_names),
        "journal": journal,
        "year": year,
        "article_type": article.get("article-type", ""),
        "licence": licence,
        "licence_url": licence_url,
        "licence_text": licence_text,
        "source_url": f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/" if pmcid else "",
        "abstract": abstract,
        "full_text": full_text,
        "word_count": len(re.findall(r"\b\w+\b", full_text)),
        "retrieval_date": RETRIEVAL_DATE,
    }

## 7. Retrieve, check and select up to ten records per theme

I fetch the candidates in batches of ten. I keep only records that pass the machine-checkable rules. I still mark every selected record as `pending_manual_review`.

In [ ]:
parsed_articles = {}
BATCH_SIZE = 10

for start in range(0, len(all_candidate_ids), BATCH_SIZE):
    batch = all_candidate_ids[start:start + BATCH_SIZE]
    xml_data = fetch_pmc_xml(batch)
    root = ET.fromstring(xml_data)
    article_elements = [root] if root.tag == "article" else root.findall(".//article")

    for article_element in article_elements:
        record = parse_article(article_element)
        if record["pmcid"]:
            parsed_articles[record["pmcid"].replace("PMC", "")] = record

    print("Retrieved batch", start // BATCH_SIZE + 1, "of", (len(all_candidate_ids) + BATCH_SIZE - 1) // BATCH_SIZE)

selected_records = []
screening_log = []
excluded_article_types = {"retraction", "correction", "expression-of-concern"}

for theme, candidate_ids in candidate_ids_by_theme.items():
    selected_for_theme = 0

    for rank, numeric_id in enumerate(candidate_ids, start=1):
        record = parsed_articles.get(numeric_id)
        reason = ""

        if record is None:
            reason = "XML record was not returned or could not be parsed"
        elif record["article_type"].lower() in excluded_article_types:
            reason = f"Excluded article type: {record['article_type']}"
        elif record["licence"] != "CC BY":
            reason = f"Licence was not confirmed as CC BY: {record['licence']}"
        elif record["word_count"] < MINIMUM_WORDS:
            reason = f"Insufficient extracted text: {record['word_count']} words"
        elif selected_for_theme >= TARGET_PER_THEME:
            reason = "Valid reserve candidate after theme target was reached"

        if reason.startswith("Valid reserve candidate"):
            status = "reserve_candidate"
        else:
            status = "excluded_by_automatic_rule" if reason else "pending_manual_review"
        log_record = {
            "theme": theme,
            "search_rank": rank,
            "numeric_pmc_id": numeric_id,
            "pmcid": record["pmcid"] if record else f"PMC{numeric_id}",
            "title": record["title"] if record else "",
            "status": status,
            "reason": reason,
        }
        screening_log.append(log_record)

        if not reason:
            selected_for_theme += 1
            selected = dict(record)
            selected.update({
                "theme": theme,
                "search_rank": rank,
                "search_query": SEARCH_QUERIES[theme],
                "screening_status": "pending_manual_review",
                "manual_exclusion_reason": "",
            })
            selected_records.append(selected)

    print(theme, "->", selected_for_theme, "records pending manual review")

print("Total records pending manual review:", len(selected_records))

## 8. Save the evidence files

I save four evidence files: the corpus manifest, full-text JSONL data, screening log and search log. The ZIP file can be downloaded from Colab and backed up with the dissertation files.

In [ ]:
def write_csv(path, rows, fieldnames):
    with path.open("w", newline="", encoding="utf-8") as file_handle:
        writer = csv.DictWriter(file_handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


manifest_fields = [
    "pmcid", "pmid", "doi", "title", "authors", "journal", "year",
    "article_type", "theme", "licence", "licence_url", "source_url", "abstract",
    "word_count", "search_rank", "search_query", "retrieval_date",
    "screening_status", "manual_exclusion_reason",
]
manifest_rows = [
    {field: record.get(field, "") for field in manifest_fields}
    for record in selected_records
]

write_csv(OUTPUT_DIRECTORY / "pmc_corpus_manifest.csv", manifest_rows, manifest_fields)
write_csv(
    OUTPUT_DIRECTORY / "pmc_screening_log.csv",
    screening_log,
    ["theme", "search_rank", "numeric_pmc_id", "pmcid", "title", "status", "reason"],
)

with (OUTPUT_DIRECTORY / "pmc_corpus.jsonl").open("w", encoding="utf-8") as file_handle:
    for record in selected_records:
        file_handle.write(json.dumps(record, ensure_ascii=False) + "\n")

with (OUTPUT_DIRECTORY / "pmc_search_log.json").open("w", encoding="utf-8") as file_handle:
    json.dump(search_log, file_handle, ensure_ascii=False, indent=2)

archive_path = shutil.make_archive(
    "/content/Adarsh_Konderu_PMC_Corpus_Stage_2",
    "zip",
    OUTPUT_DIRECTORY,
)

print("Saved manifest:", OUTPUT_DIRECTORY / "pmc_corpus_manifest.csv")
print("Saved full text:", OUTPUT_DIRECTORY / "pmc_corpus.jsonl")
print("Saved screening log:", OUTPUT_DIRECTORY / "pmc_screening_log.csv")
print("Saved search log:", OUTPUT_DIRECTORY / "pmc_search_log.json")
print("ZIP archive:", archive_path)

## 9. Download and manually screen the manifest

After running the notebook, I will download the ZIP from the Colab file panel. I will open `pmc_corpus_manifest.csv`, read each title and abstract on the linked PMC page and change `screening_status` to either `included_after_manual_review` or `excluded_after_manual_review`. Any exclusion will have a short reason.

I will not describe the automatically generated list as my final dataset until this screening is complete.

## 10. What this gives me for the next meeting

- Reproducible PMC search queries
- A dated log of the number of records found
- A balanced candidate corpus across five themes
- PMCID, PMID, DOI, title, authors, journal and year
- Article URL and machine-read licence evidence
- Extracted article text and word count
- Automatic and manual screening evidence

## What I will do after screening

My next technical step will be cleaning and chunking the included articles. I will then build embedding-based vector retrieval and compare its results with the keyword retriever used in my first prototype.